In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!unzip "/content/drive/MyDrive/EVSE.zip" -d /content/drive/MyDrive/

In [ ]:
import pandas as pd

df_net = pd.read_csv(
    "/content/drive/MyDrive/EVSE/Network Traffic/EVSE-A/csv/EVSE-A-charging-Aggressive-scan.csv"
)

print(df_net.shape)
print(df_net.columns)
df_net.head()

In [ ]:
import pandas as pd

df_net = pd.read_csv(
    "/content/drive/MyDrive/EVSE/Network Traffic/EVSE-B/csv/EVSE-B-charging-aggressive-scan.csv"
)

print(df_net.shape)
print(df_net.columns)
df_net.head()

In [ ]:
import pandas as pd
import numpy as np
import os
import glob

# Folder path
folder_path = "/content/drive/MyDrive/EVSE/Network Traffic/EVSE-B/csv"

# Get all CSV files
all_files = glob.glob(os.path.join(folder_path, "*.csv"))

print("Total files found:", len(all_files))

# Combine all files
df_list = []

for file in all_files:
    temp_df = pd.read_csv(file)

    # Create label from filename
    filename = os.path.basename(file).lower()

    if "benign" in filename:
        temp_df["Target"] = 0
    else:
        temp_df["Target"] = 1

    df_list.append(temp_df)

df = pd.concat(df_list, ignore_index=True)

print("Combined dataset shape:", df.shape)

Total files found: 31


Combined dataset shape: (2196846, 87)


In [ ]:
# Remove leakage columns safely
leakage_keywords = ['scenario', 'label', 'attack', 'type', 'class']

columns_to_drop = []

for col in df.columns:
    for keyword in leakage_keywords:
        if keyword in col.lower():
            columns_to_drop.append(col)
            break

print("Dropping columns:", columns_to_drop)

df = df.drop(columns=columns_to_drop, errors='ignore')

In [ ]:
y = df["Target"]
X = df.drop(columns=["Target"])

# Keep only numeric columns
X = X.select_dtypes(include=[np.number])

# Remove NaNs
X = X.replace([np.inf, -np.inf], np.nan)
X = X.dropna()
y = y.loc[X.index]

print("Final feature shape:", X.shape)

Final feature shape: (2196846, 73)


In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

# Remove NaNs
X = X.dropna()
y = y.loc[X.index]

# Keep only numeric columns
X = X.select_dtypes(include=[np.number])

# Standardize
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Stratified split (VERY IMPORTANT)
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print("Train size:", X_train.shape)
print("Test size:", X_test.shape)

Train size: (1757476, 73)
Test size: (439370, 73)


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

log_model = LogisticRegression(class_weight='balanced', max_iter=1000)
log_model.fit(X_train, y_train)

y_pred_log = log_model.predict(X_test)

print("Logistic Regression Results")
print("Accuracy:", accuracy_score(y_test, y_pred_log))
print(confusion_matrix(y_test, y_pred_log))
print(classification_report(y_test, y_pred_log))

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=200,
    class_weight='balanced',
    random_state=42
)

rf_model.fit(X_train, y_train)

y_pred_rf = rf_model.predict(X_test)

print("Random Forest Results")
print("Accuracy:", accuracy_score(y_test, y_pred_rf))
print(confusion_matrix(y_test, y_pred_rf))
print(classification_report(y_test, y_pred_rf))

Random Forest Results
Accuracy: 1.0
[[439370]]
              precision    recall  f1-score   support

           1       1.00      1.00      1.00    439370

    accuracy                           1.00    439370
   macro avg       1.00      1.00      1.00    439370
weighted avg       1.00      1.00      1.00    439370



In [ ]:
from xgboost import XGBClassifier

xgb_model = XGBClassifier(
    scale_pos_weight = len(y_train[y_train==0]) / len(y_train[y_train==1]),
    eval_metric='logloss',
    random_state=42
)

xgb_model.fit(X_train, y_train)

y_pred_xgb = xgb_model.predict(X_test)

print("XGBoost Results")
print("Accuracy:", accuracy_score(y_test, y_pred_xgb))
print(confusion_matrix(y_test, y_pred_xgb))
print(classification_report(y_test, y_pred_xgb))

In [ ]:
from sklearn.metrics import roc_auc_score

print("RF AUC:", roc_auc_score(y_test, rf_model.predict_proba(X_test)[:,1]))
print("XGB AUC:", roc_auc_score(y_test, xgb_model.predict_proba(X_test)[:,1]))

In [ ]:
leakage_keywords = [
    'scenario', 'label', 'attack', 'type',
    'class', 'state', 'service', 'scan'
]

drop_cols = []

for col in df.columns:
    for key in leakage_keywords:
        if key in col.lower():
            drop_cols.append(col)
            break

df = df.drop(columns=drop_cols, errors='ignore')

print("After dropping leakage:", df.shape)

In [ ]:
allowed_keywords = [
    'packet', 'byte', 'length', 'duration',
    'iat', 'flag', 'rate', 'window', 'ttl'
]

selected_cols = []

for col in df.columns:
    for key in allowed_keywords:
        if key in col.lower():
            selected_cols.append(col)
            break

selected_cols.append("Target")

df = df[selected_cols]

print("After aggressive filtering:", df.shape)

In [ ]:
# Keep only numeric
df = df.select_dtypes(include=[np.number])

# Remove NaNs
df = df.replace([np.inf, -np.inf], np.nan)
df = df.dropna()

y = df["Target"]
X = df.drop(columns=["Target"])

# Correlation with target
corr = X.corrwith(y).abs()

# Select top 15 only
top_features = corr.sort_values(ascending=False).head(15).index

X = X[top_features]

print("Final feature count:", len(top_features))
print("Selected features:", list(top_features))

Final feature count: 15
Selected features: ['bidirectional_duration_ms', 'bidirectional_packets', 'bidirectional_bytes', 'src2dst_duration_ms', 'src2dst_packets', 'src2dst_bytes', 'dst2src_duration_ms', 'dst2src_packets', 'dst2src_bytes', 'bidirectional_min_piat_ms', 'bidirectional_mean_piat_ms', 'bidirectional_stddev_piat_ms', 'bidirectional_max_piat_ms', 'src2dst_min_piat_ms', 'src2dst_mean_piat_ms']


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

model = LogisticRegression(class_weight='balanced', max_iter=1000)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

In [31]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.6,
    stratify=y,
    random_state=42
)

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

rf = RandomForestClassifier(
    n_estimators=80,        # not too high
    max_depth=6,             # restrict depth
    min_samples_split=10,
    min_samples_leaf=5,
    max_features='sqrt',
    class_weight='balanced',
    random_state=42
)

rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

Accuracy: 1.0



Confusion Matrix:
 [[1318108]]

Classification Report:
               precision    recall  f1-score   support

           1       1.00      1.00      1.00   1318108

    accuracy                           1.00   1318108
   macro avg       1.00      1.00      1.00   1318108
weighted avg       1.00      1.00      1.00   1318108

